# Information Theory Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Information content and entropy

In [ ]:
```python

import math

def information_content(p, base=2):

    if p <= 0 or p > 1:

        return float('inf') if p <= 0 else 0.0

    return -math.log(p) / math.log(base)

def entropy(probs, base=2):

    return sum(

        p * information_content(p, base)

        for p in probs if p > 0

    )

fair_coin = [0.5, 0.5]

biased_coin = [0.99, 0.01]

fair_die = [1/6] * 6

print(f"Fair coin entropy:   {entropy(fair_coin):.4f} bits")

print(f"Biased coin entropy: {entropy(biased_coin):.4f} bits")

print(f"Fair die entropy:    {entropy(fair_die):.4f} bits")

In [ ]:
```

### Step 2: Cross-entropy and KL divergence

In [ ]:
```python

def cross_entropy(p, q, base=2):

    total = 0.0

    for pi, qi in zip(p, q):

        if pi > 0:

            if qi <= 0:

                return float('inf')

            total += pi * (-math.log(qi) / math.log(base))

    return total

def kl_divergence(p, q, base=2):

    return cross_entropy(p, q, base) - entropy(p, base)

true_dist = [0.7, 0.2, 0.1]

good_model = [0.6, 0.25, 0.15]

bad_model = [0.1, 0.1, 0.8]

print(f"Entropy of true dist:     {entropy(true_dist):.4f} bits")

print(f"CE (good model):          {cross_entropy(true_dist, good_model):.4f} bits")

print(f"CE (bad model):           {cross_entropy(true_dist, bad_model):.4f} bits")

print(f"KL divergence (good):     {kl_divergence(true_dist, good_model):.4f} bits")

print(f"KL divergence (bad):      {kl_divergence(true_dist, bad_model):.4f} bits")

In [ ]:
```

### Step 3: Cross-entropy as classification loss

In [ ]:
```python

def softmax(logits):

    max_logit = max(logits)

    exps = [math.exp(z - max_logit) for z in logits]

    total = sum(exps)

    return [e / total for e in exps]

def cross_entropy_loss(true_class, logits):

    probs = softmax(logits)

    return -math.log(probs[true_class])

logits = [2.0, 1.0, 0.1]

true_class = 0

probs = softmax(logits)

loss = cross_entropy_loss(true_class, logits)

print(f"Logits:      {logits}")

print(f"Softmax:     {[f'{p:.4f}' for p in probs]}")

print(f"True class:  {true_class}")

print(f"Loss:        {loss:.4f} nats")

print(f"Perplexity:  {math.exp(loss):.2f}")

In [ ]:
```

### Step 4: Cross-entropy equals negative log-likelihood

In [ ]:
```python

import random

random.seed(42)

n_samples = 1000

n_classes = 3

true_labels = [random.randint(0, n_classes - 1) for _ in range(n_samples)]

model_logits = [[random.gauss(0, 1) for _ in range(n_classes)] for _ in range(n_samples)]

ce_loss = sum(

    cross_entropy_loss(label, logits)

    for label, logits in zip(true_labels, model_logits)

) / n_samples

nll = -sum(

    math.log(softmax(logits)[label])

    for label, logits in zip(true_labels, model_logits)

) / n_samples

print(f"Cross-entropy loss:      {ce_loss:.6f}")

print(f"Negative log-likelihood: {nll:.6f}")

print(f"Difference:              {abs(ce_loss - nll):.2e}")

In [ ]:
```

### Step 5: Mutual information

In [ ]:
```python

def mutual_information(joint_probs, base=2):

    rows = len(joint_probs)

    cols = len(joint_probs[0])

    margin_x = [sum(joint_probs[i][j] for j in range(cols)) for i in range(rows)]

    margin_y = [sum(joint_probs[i][j] for i in range(rows)) for j in range(cols)]

    mi = 0.0

    for i in range(rows):

        for j in range(cols):

            pxy = joint_probs[i][j]

            if pxy > 0:

                mi += pxy * math.log(pxy / (margin_x[i] * margin_y[j])) / math.log(base)

    return mi

independent = [[0.25, 0.25], [0.25, 0.25]]

dependent = [[0.45, 0.05], [0.05, 0.45]]

print(f"MI (independent): {mutual_information(independent):.4f} bits")

print(f"MI (dependent):   {mutual_information(dependent):.4f} bits")

In [ ]:
```

## Exercises

In [ ]:
1. Compute the entropy of the English alphabet assuming uniform distribution (26 letters). Then estimate it using actual letter frequencies. Which is higher and why?

2. A model outputs logits [5.0, 2.0, 0.5] for a sample with true class 1. Compute the cross-entropy loss by hand, then verify with your `cross_entropy_loss` function. What logits would give zero loss?

3. Show that KL divergence is not symmetric. Pick two distributions P and Q and compute D_KL(P || Q) and D_KL(Q || P). Explain why they differ.

4. Build a function that computes perplexity for a sequence of token predictions. Given a list of (true_token_index, predicted_logits) pairs, return the perplexity of the sequence.